# 🧪 Taller - Cámara en Vivo: Captura y Procesamiento de Video en Tiempo Real con YOLO

## 📅 Fecha
`2025-06-20`

---
## 🎯 Objetivo del Taller
Conectar la cámara web del PC y procesar el video en tiempo real usando Python, OpenCV y YOLO para aplicar filtros visuales y realizar detección de objetos en vivo. Este taller combina técnicas de visión artificial clásica con modelos de detección basados en aprendizaje profundo.

## 🧠 Conceptos Aprendidos
- Captura de video en tiempo real con `cv2.VideoCapture`
- Aplicación de filtros clásicos: escala de grises, binarización, bordes
- Uso de modelos YOLOv8 para detección de objetos
- Dibujar cajas, etiquetas y confianza sobre el video en vivo
- Controles con teclado para manipular la visualización
- Gestión de múltiples ventanas sincronizadas en OpenCV
- Grabación condicional de imágenes y clips de video
- Lógica condicional basada en los objetos detectados (conteo y respuesta)

## 🔧 Herramientas Usadas
- Python 3.10+
- OpenCV (`opencv-python`)
- NumPy
- YOLOv8 (`ultralytics`)
- Cámara Web

In [2]:
import cv2
from ultralytics import YOLO
import time

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)
filtro_actual = 0  # 0: original, 1: gris, 2: binario, 3: bordes
pausado = False
grabando = False
out = None

while True:
    if not pausado:
        ret, frame = cap.read()
        if not ret:
            break

        resultados = model(frame)[0]
        frame_yolo = frame.copy()
        celular_detectado = False  # ← bandera para detectar celular

        for r in resultados.boxes:
            x1, y1, x2, y2 = map(int, r.xyxy[0])
            cls = model.names[int(r.cls[0])]
            conf = r.conf[0].item()
            cv2.rectangle(frame_yolo, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame_yolo, f'{cls} {conf:.2f}', (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            # 🟡 Si se detecta un celular, activa la bandera
            if cls.lower() in ["cell phone", "mobile phone", "telefono celular"]:
                celular_detectado = True

        # 🔄 Cambiar filtro automáticamente si se detecta un celular
        if celular_detectado:
            filtro_actual = (filtro_actual + 1) % 4

        # 🎨 Aplicar el filtro correspondiente
        if filtro_actual == 1:
            filtro = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            filtro = cv2.cvtColor(filtro, cv2.COLOR_GRAY2BGR)
        elif filtro_actual == 2:
            gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            _, filtro = cv2.threshold(gris, 127, 255, cv2.THRESH_BINARY)
            filtro = cv2.cvtColor(filtro, cv2.COLOR_GRAY2BGR)
        elif filtro_actual == 3:
            gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            filtro = cv2.Canny(gris, 100, 200)
            filtro = cv2.cvtColor(filtro, cv2.COLOR_GRAY2BGR)
        else:
            filtro = frame.copy()

        cv2.imshow("Detección con YOLO", frame_yolo)
        cv2.imshow("Filtro", filtro)

        if grabando:
            out.write(frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('f'):
        filtro_actual = (filtro_actual + 1) % 4
    elif key == ord('p'):
        pausado = not pausado
    elif key == ord('s'):
        cv2.imwrite("captura.png", frame)
    elif key == ord('v') and not grabando:
        out = cv2.VideoWriter('video.avi', cv2.VideoWriter_fourcc(*'XVID'), 20.0,
                             (frame.shape[1], frame.shape[0]))
        grabando = True
        inicio = time.time()
    if grabando and time.time() - inicio > 5:
        grabando = False
        out.release()

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 257.4ms
Speed: 17.2ms preprocess, 257.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 149.9ms
Speed: 3.0ms preprocess, 149.9ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 140.8ms
Speed: 2.3ms preprocess, 140.8ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 138.1ms
Speed: 3.5ms preprocess, 138.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 129.8ms
Speed: 3.0ms preprocess, 129.8ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 128.4ms
Speed: 2.1ms preprocess, 128.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 132.2ms
Speed: 2.7ms preprocess, 132.2ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 152.5ms
Speed: 1.9ms preprocess, 152.5ms inference, 4.7ms postprocess per image a